In [1]:
import os
# Optional: set MOSEK license if needed on the remote machine
# Optional: set MOSEKLM_LICENSE_FILE to your local MOSEK license path if using MOSEK.

import numpy as np
import matplotlib.pyplot as plt

from qer.codewords import bgmcode_piqs
from qer.noisemodel import noisemodel
from qer.optimisation import optimise

# Sweep points
gamma_vals = np.array([1e-5, 5e-5, 1e-4, 5e-4, 1e-3, 5e-3])
dt = 1.0
method = "choi"  # "global symmetric depolarizing" supports "choi" in your code

# Supply larger BGM codes here
codes  = [(3, 3, 1), (5,5,2), (7,7,3)]  # e.g., [(5, 5, 2), (7, 7, 3)]
labels = [f"bgm({b},{g},{m})" for (b, g, m) in codes]

fids = {lab: [] for lab in labels}

for (b, g, m), lab in zip(codes, labels):
    N = 2 * b * m + g
    rho, l0, l1 = bgmcode_piqs(b, g, m, return_qutip=True)

    for gamma in gamma_vals:
        try:
            kraus_or_choi = noisemodel("global symmetric depolarizing", N, gamma, dt, method)
            F = float(optimise(l0, l1, kraus_or_choi, solver="mosek"))
        except Exception as e:
            print(f"{lab}: N={N}, gamma={gamma:.1e} -> ERROR: {e}")
            F = np.nan
        fids[lab].append(F)
        print(f"{lab}: N={N}, gamma={gamma:.1e} -> F_opt={F:.6f}")

# Plot infidelity vs p = gamma*dt (log-log)
p_vals = gamma_vals * dt
plt.figure(figsize=(6, 4), dpi=130)
for lab in labels:
    F = np.array(fids[lab], dtype=float)
    infid = np.abs(1.0 - F)
    mask = (infid > 0) & np.isfinite(infid)
    plt.loglog(p_vals[mask], infid[mask], "o-", label=lab)

plt.xlabel(r"error probability $p=\gamma \Delta t$")
plt.ylabel(r"$|1-F|$")
plt.title("Global symmetric depolarising: BGM codes")
plt.grid(True, which="both", ls="--", alpha=0.4)
plt.legend()
plt.tight_layout()
plt.show()

bgm(3,3,1): N=9, gamma=1.0e-05 -> ERROR: The solver MOSEK is not installed.
bgm(3,3,1): N=9, gamma=1.0e-05 -> F_opt=nan
bgm(3,3,1): N=9, gamma=5.0e-05 -> ERROR: The solver MOSEK is not installed.
bgm(3,3,1): N=9, gamma=5.0e-05 -> F_opt=nan
bgm(3,3,1): N=9, gamma=1.0e-04 -> ERROR: The solver MOSEK is not installed.
bgm(3,3,1): N=9, gamma=1.0e-04 -> F_opt=nan
bgm(3,3,1): N=9, gamma=5.0e-04 -> ERROR: The solver MOSEK is not installed.
bgm(3,3,1): N=9, gamma=5.0e-04 -> F_opt=nan
bgm(3,3,1): N=9, gamma=1.0e-03 -> ERROR: The solver MOSEK is not installed.
bgm(3,3,1): N=9, gamma=1.0e-03 -> F_opt=nan
bgm(3,3,1): N=9, gamma=5.0e-03 -> ERROR: The solver MOSEK is not installed.
bgm(3,3,1): N=9, gamma=5.0e-03 -> F_opt=nan


: 